# PySpark introduction 

PySpark is a python interface to Apache Spark which is a system for processing large datasets in parallel. It makes it possible to analyze and transform big data across multiple machines. Normal python code becomes distributed behind the scenes making it fast and scalable 

## Row object in pyspark

a Row is a single record in a DataFrame
- a structured tuple with named fields


In [0]:
from pyspark.sql import Row 

row1 = Row(name = "Bella", animal_type = "rabbit", age = 4)
row1

In [0]:
row2 = Row(name = "Doggu", animal_type = "dog", age = 10)
row2

## Create a Spark DataFrame 

Can be created from  
- Row
- list of dicts
- from csv file
- from a table 
- from an existing dataframe
- from database

We won't show all these in this demos though, but feel free to explore 

Spark dataframes are evaluated lazily meaning all the transformations are not executed until an `action` is called


In [0]:
rows = [row1, row2] 
print(rows)

df = spark.createDataFrame(rows)
df # not evaluated yet 


In [0]:
# action 
df.show()

In [0]:
# we see the constituents are Row objects
df.take(1)

In [0]:
employees = [
  {"name": "John D.", "age": 30, "department": "HR"},
  {"name": "Alice G.", "age": 25, "department": "Finance"},
  {"name": "Bob T.", "age": 35, "department": "IT"},
  {"name": "Eve A.", "age": 28, "department": "Marketing"}
]

df_employees = spark.createDataFrame(employees)
# note Row instances even though we create from list of dicts
df_employees.take(2)

### Some actions to evaluate the dataframe

In [0]:
df_employees.show(2)

In [0]:
display(df_employees)

## Data types in PySpark

Data types in pyspark are important to define DataFrame schemas for efficient data processing. DataFrames can infer the schema from data, but that costs in performance (especially for larger datasets) as spark needs to scan through each data to find apropriate data type to infer

Below are all the types in pyspark

| Type        | Description                                      |
|-------------|--------------------------------------------------|
| ByteType    | 1-byte integer (-128 to 127)                     |
| ShortType   | 2-byte integer (-32,768 to 32,767)               |
| IntegerType | 4-byte integer                                  |
| LongType    | 8-byte integer                                  |
| FloatType   | Single precision floating point                  |
| DoubleType  | Double precision floating point                  |
| DecimalType | Fixed precision decimal numbers                  |
| StringType  | Text/string values                              |
| BooleanType | True or False                                   |
| DateType    | Date (year, month, day)                         |
| TimestampType | Date and time                                  |
| BinaryType  | Binary data                                     |
| ArrayType   | List/array of elements                          |
| MapType     | Key-value pairs                                 |
| StructType  | Complex type (like a nested row / schema)       |

### Read from a csv file

- [data source](https://www.kaggle.com/datasets/heesoo37/120-years-of-olympic-history-athletes-and-results)

Take note of the time here when spark infers the schema

In [0]:

from pathlib import Path
DATA_PATH = Path().resolve() / "data"

df_athletes = spark.read.csv(str(DATA_PATH /"athlete_events.csv"), header=True)

display(df_athletes)

In [0]:
df_athletes.printSchema()

In [0]:
print(df_athletes.columns)

read from same csv file with defined schema

In [0]:
from pyspark.sql.types import StructField, StringType, ShortType, ByteType, StructType, IntegerType

schema = StructType([
  StructField("ID", IntegerType()),
  StructField("Name", StringType()),
  StructField("Sex", StringType()),
  StructField("Age", ByteType()),
  StructField("Height", ShortType()),
  StructField("Weight", ShortType()),
  StructField("Team", StringType()),
  StructField("NOC", StringType()),
  StructField("Games", StringType()),
  StructField("Year", ShortType()),
  StructField("Season", StringType()),
  StructField("City", StringType()),
  StructField("Sport", StringType()),
  StructField("Event", StringType()),
  StructField("Medal", StringType())
])

df_athletes_schema = spark.read.csv(str(DATA_PATH/"athlete_events.csv"), header=True, schema=schema)
display(df_athletes_schema)


Faster and the schema makes more sense than just string that was inferred. Note that all values that are mistype and cannot be cast to correct type becomes null here instead

In [0]:
df_athletes_schema.printSchema()

Find nulls per columns

In [0]:
from pyspark.sql.functions import col, sum

nulls = df_athletes_schema.select(
    [sum(col(c).isNull().cast("int")).alias(c) for c in df_athletes_schema.columns]
)
display(nulls)

In [0]:
df_athletes_schema.groupBy("NOC").count().filter("NOC = 'SWE'").show()

In [0]:
df_athletes_schema.createOrReplaceTempView("df_athletes_schema")

df_swe_medals = spark.sql("""
          SELECT 
            sport, count(medal) AS medal_count
          FROM df_athletes_schema
          WHERE noc = 'SWE' AND medal IN ('Gold', 'Silver', 'Bronze')
          GROUP BY sport
          ORDER BY medal_count DESC
          """)

display(df_swe_medals)

In [0]:
fig = df_swe_medals.plot(kind = "bar", y = "sport", x = "medal_count", title = "Swedish medals",)
fig.update_layout(yaxis = {"autorange": "reversed"})



## SQL cells

In [0]:
%sql 
FROM df_athletes_schema
LIMIT 1

In [0]:
%sql 
DROP CATALOG IF EXISTS data CASCADE;

## Export selection of data to table in Unity catalog

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS data;

CREATE SCHEMA IF NOT EXISTS data.olympics;

CREATE TABLE IF NOT EXISTS data.olympics.sweden_medals AS
(
  SELECT
    name,
    age,
    weight,
    height,
    year,
    sport,
    medal
  FROM
    df_athletes_schema
  WHERE
    noc = 'SWE'
    and medal IN ('Gold', 'Bronze', 'Silver')
);

In [0]:
%sql

FROM data.olympics.sweden_medals;